From Fakeddit paper: 
"In addition, we used the BERT model. BERT achieves state-of-the-art results on many classification tasks, including Q&A and named entity recognition. To obtain fixed-length BERT embedding vectors, we used the bert-as-service(Xiao, 2018) tool, to map variable-length text/sentences into a 768 element array for each Reddit submission title. For our experiments, we utilized the pretrained BERT-Large, Uncased model."

Paper achieves around 82% accurcy on 3-way labels, so I should aim for the same

Use original BERT paper (Devlin et al., 2019) for fine-tuning procedure. Can be found in appendix A3:
- Batch size: 16 or 32
- Num epochs: 2, 3, or 4
- Select best learning rate from 2, 3, 4 or 5 e-5 on val set
- Dropout: 0.1
- Large datasets (>100K labelled examples) far less sensitive to hyperparameter choice than small.
- Fine-tuning is fast - best to run an exhaustive search across the above hyperparameters on the train/val set and see which is best. 

Bert github code provides more set up details (google-research/bert):
- AdamW optimiser (β1=0.9, β2=0.999, ε=1e-6)
- Weight decay = 0.01
- Warmup ratio (10% of total steps in published code)

(Mouratidis et al., 2025) - good paper for supporting claim that fine-tined BERT is current best practice for fake new detection. Not so good for fine-tuning methodology but they do have some interesting stuff to take into account: 
- They use Matthews Correlation Coefficient (MCC) and ROC_AUC alongside classic accuracy and macro F1 - good for datasets with a class imbalance (like mine). 
- Find that non-stemmed text performs better than stemmed and that unigrams are sufficient 


This method is a bit outdated. Instead, going to fine tune `bert-base-uncased` end-to-end via Hugging Face Transformers library. Current best practice for BERT-based text classification.

Why `bert-base-uncased`?
- Save GPU - BERT Large requires more GPU< likely for minimal gain. Use BERT-base instead
- Uncased - using Reddit titles with inconsistent casing. Since Fakeddit's clean_title col is already lowercased, uncased makes the most sense. 
- Example explanation: "We use bert-base-uncased (Devlin et al., 2019: L=12, H=768, A=12, 110M parameters) rather than BERT-Large, balancing classification performance against training time within the project's compute budget. Devlin et al. (2019, §5.2) report that BERT-Large outperforms Base on most tasks but at substantially higher computational cost; for our classification setting on short titles, base-size models are standard in recent comparable work (Mouratidis et al., 2025; numerous HuggingFace baselines)."


In [13]:
import os
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import itertools
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score

In [8]:
data_dir = os.path.join('..', 'data', 'processed', 'US')

train_path = os.path.join(data_dir, 'multimodal_train.tsv')
val_path = os.path.join(data_dir, 'multimodal_validate.tsv')

train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')

In [3]:
MODEL_NAME = 'bert-base-uncased'
NUM_LABELS = 3
MAX_LEN = 32 # from EDA, at least 75% of titles are under 10 words


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def tokenise(batch):
    return tokenizer(batch['clean_title'], truncation=True, padding='max_length', max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df).map(tokenise, batched=True).rename_column('3_way_label', 'labels')
val_ds = Dataset.from_pandas(val_df).map(tokenise, batched=True).rename_column('3_way_label', 'labels')

Map:   0%|          | 0/564000 [00:00<?, ? examples/s]

Map:   0%|          | 0/59342 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predctions, axis=1)
    labels = eval_pred.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1 score': f1_score(labels, preds, average='macro'),
        'MCC': matthews_corrcoef(labels, preds)
    }

As recommended in BERT appendix A3, can do an exhaustive grid search over their batch size and lr hyperparamter recommendations

In [16]:
BATCH_SIZES = [16, 32]
LEARNING_RATES = [5e-5, 3e-5, 2e-5]
MAX_EPOCHS = 4

results = []

for batch_size, lr in itertools.product(BATCH_SIZES, LEARNING_RATES):
    loop_name = f'bs_{batch_size}_lr_{lr}'
    print(f'training run: {loop_name}')

    training_args = TrainingArguments(
        output_dir=f'../models/bert_finetuned/{loop_name}',
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,        # only keep best checkpoint to save disk
        report_to="none",
        logging_strategy="epoch",
        seed=42,   
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    best_metrics = trainer.evaluate()
    
    results.append({
        'run_name': loop_name,
        "batch_size": batch_size,
        'learning_rate': lr,
        'best_epoch': trainer.state.best_metric and trainer.state.epoch **best_metrics,
    })

    # save results after each run
    with open('../results/bert_grid_search.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


training run: bs_16_lr_5e-05


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [17]:
# summarise results in df
results_df = pd.DataFrame(results).sort_values('eval_macro_f1', ascending=False)
print(df[["run_name", "eval_macro_f1", "eval_accuracy", "eval_mcc"]].to_string())
print(f"\nBest config: {df.iloc[0]['run_name']}")

KeyError: 'eval_macro_f1'